# Deep Learning Colorization Pipeline

**Method**: Zhang et al. 2016 - "Colorful Image Colorization" (ECCV)

This notebook implements the full deep learning colorization pipeline:
1. Data exploration and Lab color space visualization
2. Model architecture (Zhang16Net - reimplemented in PyTorch)
3. Training / fine-tuning on COCO 2017
4. Evaluation with PSNR, SSIM, LPIPS
5. Comparison with pre-trained models: DeOldify (GAN) and ControlNet (Diffusion)

In [ ]:
# Setup and imports
import sys
import os
import warnings
warnings.filterwarnings("ignore")

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from skimage import color as skcolor

from src.deep_learning.utils import load_config, rgb_to_lab, lab_to_rgb, compute_metrics, visualize_result, get_device
from src.deep_learning.quantize import ABQuantizer
from src.deep_learning.model import Zhang16Net, Zhang16Regression, build_model
from src.deep_learning.loss import build_loss
from src.deep_learning.dataset import ColorizationDataset, get_dataloaders
from src.deep_learning.colorizer import DeepColorizer
from src.deep_learning.train import Trainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load config
cfg = load_config("../configs/config.yaml")
device = get_device(cfg)
print(f"Device: {device}")
print(f"Model: {cfg['deep_learning']['model']}")
print(f"Loss: {cfg['deep_learning']['loss']}")
print(f"Epochs: {cfg['deep_learning']['epochs']}")
print(f"Batch size: {cfg['deep_learning']['batch_size']}")
print(f"Learning rate: {cfg['deep_learning']['learning_rate']}")

## 1. Data Exploration - CIE Lab Color Space

The core idea: instead of predicting RGB directly, we work in **CIE Lab** color space.
- **L channel**: Lightness (grayscale) - this is our input
- **a, b channels**: Color information - this is what the model predicts

This separation allows us to frame colorization as: given L, predict (a, b).

In [ ]:
# Create a synthetic test image to demonstrate Lab color space
np.random.seed(42)

# Create a colorful image with 4 quadrants
h, w = 256, 256
img = np.zeros((h, w, 3), dtype=np.uint8)
img[:h//2, :w//2] = [255, 0, 0]       # Red
img[:h//2, w//2:] = [0, 255, 0]       # Green
img[h//2:, :w//2] = [0, 0, 255]       # Blue
img[h//2:, w//2:] = [255, 255, 0]     # Yellow

# Add gradients for more visual interest
for i in range(h):
    for j in range(w):
        img[i, j] = np.clip(img[i, j].astype(int) - abs(i - h//2) // 2 - abs(j - w//2) // 2, 0, 255)

# Convert to Lab and visualize
L, ab = rgb_to_lab(img)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img)
axes[0, 0].set_title("Original RGB")
axes[0, 1].imshow(L, cmap="gray")
axes[0, 1].set_title(f"L channel (Lightness)\nRange: [{L.min():.1f}, {L.max():.1f}]")
axes[0, 2].imshow(ab[:, :, 0], cmap="RdBu_r")
axes[0, 2].set_title(f"a channel (Green-Red)\nRange: [{ab[:,:,0].min():.1f}, {ab[:,:,0].max():.1f}]")

axes[1, 0].imshow(ab[:, :, 1], cmap="RdBu_r")
axes[1, 0].set_title(f"b channel (Blue-Yellow)\nRange: [{ab[:,:,1].min():.1f}, {ab[:,:,1].max():.1f}]")

# Reconstruct from Lab
reconstructed = lab_to_rgb(L, ab)
axes[1, 1].imshow(reconstructed)
axes[1, 1].set_title("Reconstructed from Lab")

# Grayscale (what model sees as input)
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
axes[1, 2].imshow(gray, cmap="gray")
axes[1, 2].set_title("Grayscale (Model Input)")

for ax in axes.flat:
    ax.axis("off")
plt.suptitle("CIE Lab Color Space Decomposition", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Ab Color Quantization

Zhang et al. 2016 treats colorization as a **classification problem** over 
quantized ab color bins, not a regression problem.

The ab space [-110, 110] is divided into a 10x10 grid, and only in-gamut 
bins are kept. This avoids the "desaturation" problem of MSE regression.

In [ ]:
# Visualize the ab quantization bins
quantizer = ABQuantizer()
print(f"Number of in-gamut ab bins: {quantizer.num_bins}")
print(f"Ab bins shape: {quantizer.ab_bins.shape}")

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.scatter(quantizer.ab_bins[:, 0], quantizer.ab_bins[:, 1], 
           c=range(quantizer.num_bins), cmap="hsv", s=30, alpha=0.8)
ax.set_xlabel("a (Green-Red)")
ax.set_ylabel("b (Blue-Yellow)")
ax.set_title(f"Quantized ab Color Bins ({quantizer.num_bins} in-gamut bins)")
ax.set_xlim(-120, 120)
ax.set_ylim(-120, 120)
ax.grid(True, alpha=0.3)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 3. Model Architecture

The Zhang16Net uses 8 convolutional blocks with:
- **Blocks 1-3**: Standard convolutions with stride-2 downsampling (H -> H/2 -> H/4 -> H/8)
- **Blocks 4-7**: Dilated convolutions at H/8 resolution (larger receptive field without losing spatial info)
- **Block 8**: Transposed convolution to upsample to H/4, then 1x1 conv to predict class probabilities

Total ~31.6M parameters. Output is upsampled to full resolution at inference time.

In [ ]:
# Build model and inspect architecture
model = build_model(cfg, quantizer=quantizer)
print(f"Model: {type(model).__name__}")
print(f"Output classes: {quantizer.num_bins}")

# Parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
x = torch.randn(1, 1, 256, 256)
with torch.no_grad():
    logits = model(x)
    ab_pred = model.predict_ab(x, quantizer, temperature=0.38)
print(f"\nForward pass test:")
print(f"  Input:  {x.shape}")
print(f"  Logits: {logits.shape}  (B, {quantizer.num_bins}, H/4, W/4)")
print(f"  Ab:     {ab_pred.shape}  (B, 2, H, W)")

## 4. Quick Colorization Test (Untrained Model)

Before training, let's test the full inference pipeline with random weights.
The output will be noisy/random, but this verifies the pipeline works end-to-end.

In [ ]:
# Test colorization with untrained model
colorizer = DeepColorizer(device=str(device), temperature=0.38)

# Use our synthetic test image
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
result_bgr, info = colorizer.colorize(gray)
result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

fig = visualize_result(gray, result_rgb, img, title="Untrained Model - Pipeline Test")
plt.show()
print(f"Inference time: {info['elapsed_sec']:.3f}s on {info['device']}")

## 5. Training on COCO 2017

**Prerequisites**: Run these commands first to download data:
```bash
python tools/download_coco.py --split both
python tools/download_pretrained.py --model zhang16
```

Training uses:
- **Loss**: Class-rebalanced cross-entropy (or Huber for regression variant)
- **Optimizer**: Adam (lr=2e-4)
- **Scheduler**: StepLR (decay every 20 epochs)
- **Mixed precision**: Enabled for 6GB VRAM GPUs
- **MLflow**: Tracks all hyperparameters and metrics per epoch

In [ ]:
# Load data
train_loader, val_loader, test_loader = get_dataloaders(cfg)

if train_loader is not None:
    print(f"Train: {len(train_loader.dataset)} images ({len(train_loader)} batches)")
if val_loader is not None:
    print(f"Val:   {len(val_loader.dataset)} images ({len(val_loader)} batches)")
if test_loader is not None:
    print(f"Test:  {len(test_loader.dataset)} images ({len(test_loader)} batches)")

if train_loader is None:
    print("\nNo training data found!")
    print("Run: python tools/download_coco.py --split both")

In [ ]:
# Setup training with MLflow
try:
    import mlflow
    mlflow.set_experiment("deep-colorization")
    mlflow_available = True
    print("MLflow experiment: deep-colorization")
except ImportError:
    mlflow_available = False
    print("MLflow not available - training without experiment tracking")

# Build model and loss
model = build_model(cfg, quantizer=quantizer)
loss_fn = build_loss(cfg, quantizer=quantizer)

print(f"Model: {type(model).__name__}")
print(f"Loss: {type(loss_fn).__name__}")
print(f"Device: {device}")
print(f"AMP: {cfg['deep_learning'].get('use_amp', False)}")

In [ ]:
# Train (skip if no data, or if a best checkpoint already exists).
# The notebook is a pipeline walkthrough — the live training run was done
# separately via `python tools/train_deep.py`; we don't redo 50 epochs each
# time the notebook is executed.
best_path = os.path.join("..", cfg["paths"]["models_deep"], "best_model.pth")

if train_loader is None:
    print("Skipping training - no data available")
    print("To train, run: python tools/download_coco.py --split both")
elif os.path.exists(best_path):
    print(f"Skipping training - already trained ({best_path} exists).")
    print("To re-train from scratch, delete that file and run:")
    print("  python tools/train_deep.py --config configs/config.yaml")
else:
    if mlflow_available:
        with mlflow.start_run():
            mlflow.log_params({
                "model": type(model).__name__,
                "loss": type(loss_fn).__name__,
                "epochs": cfg["deep_learning"]["epochs"],
                "batch_size": cfg["deep_learning"]["batch_size"],
                "learning_rate": cfg["deep_learning"]["learning_rate"],
                "num_classes": quantizer.num_bins,
                "device": str(device),
            })
            trainer = Trainer(model, train_loader, val_loader, loss_fn, cfg, device, quantizer=quantizer)
            trainer.fit()
            if os.path.exists(best_path):
                mlflow.log_artifact(best_path)
    else:
        trainer = Trainer(model, train_loader, val_loader, loss_fn, cfg, device, quantizer=quantizer)
        trainer.fit()


## 6. Evaluation

Load the best trained model and evaluate on the benchmark test set.
Computes PSNR, SSIM (and optionally LPIPS) metrics.

In [ ]:
# Load best model for evaluation
best_model_path = os.path.join("..", cfg["paths"]["models_deep"], "best_model.pth")

if os.path.exists(best_model_path):
    eval_colorizer = DeepColorizer(
        model_path=best_model_path, cfg=cfg, 
        device=str(device), temperature=cfg["deep_learning"]["temperature"]
    )
    print(f"Loaded trained model from {best_model_path}")
else:
    eval_colorizer = DeepColorizer(cfg=cfg, device=str(device))
    print("No trained model found - using untrained model for demo")
    print(f"Expected path: {best_model_path}")

# Evaluate on benchmark (or synthetic images if no benchmark)
benchmark_dir = os.path.join("..", cfg["paths"]["data_coco"], "benchmark")

if os.path.isdir(benchmark_dir):
    from tools.evaluate_deep import evaluate
    output_dir = os.path.join("..", cfg["paths"]["results_deep"], "metrics")
    results = evaluate(eval_colorizer, benchmark_dir, output_dir, max_images=20)
    
    psnr_vals = [r["psnr"] for r in results]
    ssim_vals = [r["ssim"] for r in results]
    print(f"\nResults on {len(results)} images:")
    print(f"  PSNR: {np.mean(psnr_vals):.2f} +/- {np.std(psnr_vals):.2f}")
    print(f"  SSIM: {np.mean(ssim_vals):.4f} +/- {np.std(ssim_vals):.4f}")
else:
    # Demo with synthetic image
    print("No benchmark data - testing with synthetic image")
    result_bgr, info = eval_colorizer.colorize(gray)
    result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)
    metrics = compute_metrics(result_rgb, img)
    print(f"  PSNR: {metrics['psnr']:.2f}, SSIM: {metrics['ssim']:.4f}")

## 7. Qualitative Results

Visualize colorization results on sample images.
Shows grayscale input, model prediction, and ground truth side-by-side.

In [ ]:
# Qualitative results grid
if os.path.isdir(benchmark_dir):
    # Use real images
    test_images = sorted(os.listdir(benchmark_dir))[:6]
    fig, axes = plt.subplots(len(test_images), 3, figsize=(12, 4 * len(test_images)))
    
    for i, img_name in enumerate(test_images):
        img_path = os.path.join(benchmark_dir, img_name)
        img_bgr = cv2.imread(img_path)
        gt_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray_test = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        
        result_bgr, _ = eval_colorizer.colorize(gray_test)
        pred_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)
        
        axes[i, 0].imshow(gray_test, cmap="gray")
        axes[i, 0].set_title("Input" if i == 0 else "")
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("Predicted" if i == 0 else "")
        axes[i, 2].imshow(gt_rgb)
        axes[i, 2].set_title("Ground Truth" if i == 0 else "")
        
        for ax in axes[i]:
            ax.axis("off")
    
    plt.suptitle("Colorization Results", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No benchmark images available. Run: python tools/download_coco.py")
    # Show synthetic demo
    fig = visualize_result(gray, result_rgb, img, title="Demo with Synthetic Image")
    plt.show()

## 8. Model Comparison

Compare our Zhang16 (CNN) model against pre-trained models from different categories:
- **GAN**: DeOldify (Self-Attention GAN / NoGAN) - industry standard for old photo restoration
  - Theory: ChromaGAN (Vitoria et al., WACV 2020)
- **Diffusion**: ControlNet + Stable Diffusion - state-of-the-art generative colorization
  - Theory: Palette (Saharia et al., CVPR 2022)

Note: Comparison models require additional dependencies:
- DeOldify: `pip install deoldify`
- ControlNet: `pip install diffusers transformers accelerate`

In [ ]:
# Load comparison models
from src.deep_learning.pretrained import get_comparison_models

comparison_models = get_comparison_models(cfg)
print(f"Available comparison models: {list(comparison_models.keys()) if comparison_models else 'None'}")

if not comparison_models:
    print("\nNo comparison models available.")
    print("To use DeOldify (GAN):      pip install deoldify")
    print("To use ControlNet (Diffusion): pip install diffusers transformers accelerate")
    print("\nComparison will only include our Zhang16 (CNN) model.")

# Run comparison on sample images
all_models = {"Zhang16 - CNN (Ours)": eval_colorizer}
all_models.update(comparison_models)

print(f"\nModels for comparison: {list(all_models.keys())}")

## Summary

| Component | Implementation |
|-----------|---------------|
| **Color Space** | CIE Lab (input: L channel, predict: ab channels) |
| **Architecture** | Zhang16Net - 8 conv blocks, dilated convolutions, ~31.6M params |
| **Loss** | Class-rebalanced cross-entropy over quantized ab bins |
| **Training** | Adam optimizer, StepLR scheduler, mixed precision |
| **Evaluation** | PSNR, SSIM, LPIPS on COCO 2017 benchmark |
| **Tracking** | MLflow experiment logging |

### Next Steps
- Run `python tools/download_coco.py` to get training data
- Run `python tools/train_deep.py` to train the model
- Run `python tools/evaluate_deep.py` to evaluate
- Run `python tools/compare_methods.py` for cross-method comparison
- See `notebooks/04_method_comparison.ipynb` for the 5-model deep-learning comparison